# Orchestrator

Примеры использования пайплайна:

In [1]:
import sys
import os
import json

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# Добавляем корень репозитория в путь, чтобы импортировать пакет app
sys.path.insert(0, os.path.abspath(".."))

from app.orchestrator.runner import run
from app.models.state import PipelineResult
from app.logger import setup_logging
from app.config import get_config
from app.generator.agent import GeneratorAgent

/Users/mishasdk/repo/mipt-secure-sql-agents/.venv/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
# Можно задать переменные окружения в .env, либо перезаписать тут.
os.environ["MODEL_NAME"] = "openai/gpt-4o-mini"

# Включаем логгирование.
setup_logging()

## Запуск пайплана целиком

In [3]:
query = "Show me all users who registered in the last 30 days"

result: PipelineResult = run(query)
result

2026-05-16 19:01:22 [INFO] app.orchestrator.runner: Pipeline started | query='Show me all users who registered in the last 30 days'
2026-05-16 19:01:22 [INFO] app.orchestrator: Building graph | model=openai/gpt-4o-mini
2026-05-16 19:01:23 [INFO] app.judge: load_policy | path=/Users/mishasdk/repo/mipt-secure-sql-agents/app/judge/res/security_rules.yaml | rules=4
2026-05-16 19:01:23 [INFO] app.generator: GeneratorAgent called | query='Show me all users who registered in the last 30 days'
2026-05-16 19:01:23 [INFO] app.generator.rag: tool=retriever | action=retrieve | query='Show me all users who registered in the last 30 days' | k=8
2026-05-16 19:01:23 [INFO] app.generator.rag: tool=embeddings | action=query | model=perplexity/pplx-embed-v1-4b
2026-05-16 19:01:24 [INFO] app.generator.rag: tool=embeddings | elapsed=1.27s
2026-05-16 19:01:24 [INFO] app.generator.rag: tool=retriever | result_tables=['participant_app', 'sys_company', 'scp_sec_check_res', 'scp_project_ans', 'mler_application'

PipelineResult(final_sql='SELECT * FROM sys_employee; /* Дополнительная структура запроса, если она была нужна */  \n/* Примечание: Предполагается, что в SQL-запросе присутствовала секция, где следует указать дополнительные условия или структуры. */\n\n/* Обратите внимание, если нужно исправить на SELECT sys_employee.*,  замените это в соответствии с требованиями. */\n\n--\n/* Необходимо добавить соответствующие условия, если они ускользнули или отсутствовали. */\n--\n--\n /* Существует ошибка SYNTAX, так как в запросе отсутствует весь контекст и структура, предоставленная schevra. Приведённое выше является иллюстративным примером, предполагатетльно чтобы вводить дальнейшие условия или секции */\n --  \n\n;  \n\n-- Точка должна быть добавлена в конце запроса чтобы избежать ошибок синтаксиса.\n/* Убедитесь в правильности формата всех остальной секций запроса. */\n/* Пример навигации по запросам должен начаться таким образом */\n/*  \n\nSELECT [columns]  \n \n*/ /* В зависимости от конте

In [ ]:
print(f"Approved: {result.approved}")
print(f"Iterations used: {result.iterations_used}")
print(f"\nFinal SQL:\n{result.final_sql}")

print("\nIteration logs:")
print(json.dumps([log.model_dump(mode="json") for log in result.iteration_logs], indent=2))

## Запуск Generator

In [ ]:
config = get_config()
llm = ChatOpenAI(
    base_url=config.openrouter_base_url,
    api_key=config.open_router_api_key,
    model=config.model_name,
)

generator = GeneratorAgent(llm)

state = {
    "messages": [HumanMessage(content="Show me all users who registered in the last 30 days")],
    "judge_responses": [],
    "llm_calls": 0,
    "audit_iteration": 0,
}

output = generator(state)
print(output)

Других агентов можно запускать по аналогии.